# COSC2670/2738 Assignment 3 — Framework

**Important:** Please read the comments carefully. Insert your code only in the designated cells.
Do **not** modify cells marked with `# Please don't change this cell`.

Changing locked cells may cause errors during automatic marking and invalidate your submission.

## Install and load necessary packages

In [8]:
# Please don't change this cell

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

## Load the MovieLens 100K dataset

You must use `u1.base` / `u1.test` provided in the dataset and use it consistently across all tasks. Do not create your own random split.


In [9]:
# You may change the file paths to use a different split pair
# e.g. 'ml-100k/u2.base' and 'ml-100k/u2.test'
TRAIN_PATH = 'ml-100k/u1.base'
TEST_PATH = 'ml-100k/u1.test'

names = ['user_id', 'item_id', 'rating', 'timestamp']
train_df = pd.read_csv(TRAIN_PATH, sep='\t', names=names)
test_df = pd.read_csv(TEST_PATH, sep='\t', names=names)

n_users = 943
n_items = 1682

print(f'{n_users} users')
print(f'{n_items} items')
print(f'Training ratings: {len(train_df)}')
print(f'Test ratings: {len(test_df)}')

943 users
1682 items
Training ratings: 80000
Test ratings: 20000


## Construct the user-item rating matrices

In [10]:
# Please don't change this cell

# Build training rating matrix (n_users x n_items), 0 means unrated
train_matrix = np.zeros((n_users, n_items))
for row in train_df.itertuples():
    train_matrix[row.user_id - 1, row.item_id - 1] = row.rating

# Build test rating matrix
test_matrix = np.zeros((n_users, n_items))
for row in test_df.itertuples():
    test_matrix[row.user_id - 1, row.item_id - 1] = row.rating

print("Training rating matrix shape:", train_matrix.shape)
print("Test rating matrix shape:", test_matrix.shape)

Training rating matrix shape: (943, 1682)
Test rating matrix shape: (943, 1682)


## MAE and RMSE evaluation utilities

In [11]:
# Please don't change this cell

EPSILON = 1e-9

def evaluate(test_matrix, predicted_matrix):
    '''
    Evaluate rating prediction using MAE and RMSE.
    Only considers entries where test_matrix > 0 (i.e. actual test ratings).
    '''
    mask = test_matrix > 0
    n_test = np.sum(mask)
    if n_test == 0:
        return 0.0, 0.0
    errors = test_matrix[mask] - predicted_matrix[mask]
    MAE = np.mean(np.abs(errors))
    RMSE = np.sqrt(np.mean(errors ** 2))
    return MAE, RMSE


---
# Task 1: Reproduce the Sarwar Item-Based CF Baseline Model (10 marks)

Implement the item-based collaborative filtering method from Sarwar et al. (2001) with:
- **Similarity:** Adjusted cosine similarity (Section 3.1.3)
- **Prediction:** KNN-regression as taught in lectures (k = 20)

Your code must:
1. Compute the adjusted cosine item-item similarity matrix using the training data
2. For each test rating, predict the rating using KNN-regression with k=20 neighbours
3. Store all predictions in a `predicted_matrix` (same shape as `train_matrix`)
4. Save the MAE and RMSE into the variables below

In [14]:
# Write your code here for Task 1
YOUR_PREDICTED_MATRIX = np.zeros((n_users, n_items))
MAE_task1,RMSE_task1 = 0,0
# np.zeros((n_users, n_items)) and 0 are initial values; you need to update this with the actual performance of your implementation.


# Implement the baseline item-based CF model:
#   - Adjusted cosine similarity for item-item similarity
#   - KNN-regression prediction with k = 20
#
# You must produce a predicted_matrix of shape (n_users, n_items)
# containing your predicted ratings.
#
# Then evaluate using the provided evaluate() function.


MAE_task1,RMSE_task1 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)

In [18]:
# Please don't change this cell
print("===================== The MAE and RMSE of Your Implementation =====================")
print("Task 1: Baseline Item-Based CF (Adjusted Cosine + KNN-Regression, k=20)")
print("MAE: {}, RMSE: {}" .format(MAE_task1, RMSE_task1))
print("=" * 70)

===================== The MAE and RMSE of Your Implementation =====================
Task 1: Baseline Item-Based CF (Adjusted Cosine + KNN-Regression, k=20)
MAE: 3.5359, RMSE: 3.719341339538494


---
# Task 2: Controlled Improvement Experiments (10 marks)

Choose **two** options from A–D below. Each experiment is independent.
Except for the specific parameter being tested, all other settings must remain the same as the baseline (adjusted cosine, KNN-regression, k=20).

**You must indicate which two options you chose.**

In [30]:
# Please specify which two options you selected (e.g. 'A', 'B', 'C', or 'D')
OPTION_1 = 'A'  # e.g. 'A'
OPTION_2 = 'B'  # e.g. 'B'

## Task 2 — Option A: Similarity Measure Comparison

Replace the baseline adjusted cosine similarity with **cosine similarity** and **Pearson correlation**.
Keep k=20 and KNN-regression. Report MAE and RMSE for each.

**Skip this section if you did not select Option A.**

In [31]:
# Write your code here for Option A (if selected)
# 1. Compute cosine similarity -> predict -> evaluate
# 2. Compute Pearson correlation similarity -> predict -> evaluate
MAE_optionA_cosine, RMSE_optionA_cosine = 0, 0
MAE_optionA_pearson, RMSE_optionA_pearson = 0, 0

# --- Save your results ---
MAE_optionA_cosine, RMSE_optionA_cosine = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionA_pearson, RMSE_optionA_pearson = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)

In [32]:
# Please don't change this cell

print("=" * 70)
print("Option A: Similarity Measure Comparison")
print("=" * 70)
print(f"Cosine similarity:      MAE = {MAE_optionA_cosine},  RMSE = {RMSE_optionA_cosine}")
print(f"Pearson correlation:    MAE = {MAE_optionA_pearson},  RMSE = {RMSE_optionA_pearson}")
print(f"Baseline (adj cosine):  MAE = {MAE_task1},  RMSE = {RMSE_task1}")

Option A: Similarity Measure Comparison
Cosine similarity:      MAE = 3.5359,  RMSE = 3.719341339538494
Pearson correlation:    MAE = 3.5359,  RMSE = 3.719341339538494
Baseline (adj cosine):  MAE = 3.5359,  RMSE = 3.719341339538494


## Task 2 — Option B: Neighbourhood Size Tuning

Test **k = 5, 10, 20, 30, 50** using the baseline similarity (adjusted cosine) and KNN-regression.
Report MAE and RMSE for each k value.

**Skip this section if you did not select Option B.**

In [33]:
# Write your code here for Option B (if selected)
# Test k = 5, 10, 20, 30, 50 and record MAE/RMSE for each

# --- Save your results ---
MAE_optionB_k5, RMSE_optionB_k5 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionB_k10, RMSE_optionB_k10 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionB_k20, RMSE_optionB_k20 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionB_k30, RMSE_optionB_k30 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionB_k50, RMSE_optionB_k50 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)

In [34]:
# Please don't change this cell

print("=" * 70)
print("Option B: Neighbourhood Size Tuning")
print("=" * 70)
print(f"k =  5:  MAE = {MAE_optionB_k5},  RMSE = {RMSE_optionB_k5}")
print(f"k = 10:  MAE = {MAE_optionB_k10},  RMSE = {RMSE_optionB_k10}")
print(f"k = 20:  MAE = {MAE_optionB_k20},  RMSE = {RMSE_optionB_k20}")
print(f"k = 30:  MAE = {MAE_optionB_k30},  RMSE = {RMSE_optionB_k30}")
print(f"k = 50:  MAE = {MAE_optionB_k50},  RMSE = {RMSE_optionB_k50}")

Option B: Neighbourhood Size Tuning
k =  5:  MAE = 3.5359,  RMSE = 3.719341339538494
k = 10:  MAE = 3.5359,  RMSE = 3.719341339538494
k = 20:  MAE = 3.5359,  RMSE = 3.719341339538494
k = 30:  MAE = 3.5359,  RMSE = 3.719341339538494
k = 50:  MAE = 3.5359,  RMSE = 3.719341339538494


## Task 2 — Option C: Significance Weighting

Apply significance weighting: `weighted_sim(i,j) = sim(i,j) × min(c(i,j), T) / T`

Test **T = 10, 20, 30, 50**. Keep adjusted cosine, KNN-regression, k=20.
Report MAE and RMSE for each T value.

**Skip this section if you did not select Option C.**

In [35]:
# Write your code here for Option C (if selected)
# Test T = 10, 20, 30, 50 and record MAE/RMSE for each



# --- Save your results ---
MAE_optionC_T10, RMSE_optionC_T10 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionC_T20, RMSE_optionC_T20 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionC_T30, RMSE_optionC_T30 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionC_T50, RMSE_optionC_T50 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)

In [36]:
# Please don't change this cell

print("=" * 70)
print("Option C: Significance Weighting")
print("=" * 70)
print(f"T = 10:  MAE = {MAE_optionC_T10},  RMSE = {RMSE_optionC_T10}")
print(f"T = 20:  MAE = {MAE_optionC_T20},  RMSE = {RMSE_optionC_T20}")
print(f"T = 30:  MAE = {MAE_optionC_T30},  RMSE = {RMSE_optionC_T30}")
print(f"T = 50:  MAE = {MAE_optionC_T50},  RMSE = {RMSE_optionC_T50}")
print(f"Baseline (no weighting):  MAE = {MAE_task1},  RMSE = {RMSE_task1}")

Option C: Significance Weighting
T = 10:  MAE = 3.5359,  RMSE = 3.719341339538494
T = 20:  MAE = 3.5359,  RMSE = 3.719341339538494
T = 30:  MAE = 3.5359,  RMSE = 3.719341339538494
T = 50:  MAE = 3.5359,  RMSE = 3.719341339538494
Baseline (no weighting):  MAE = 3.5359,  RMSE = 3.719341339538494


## Task 2 — Option D: Prediction Method Comparison

Replace KNN-regression with the **weighted sum** method (Section 3.2.1 of Sarwar et al.).
Keep adjusted cosine similarity and k=20.
Report MAE and RMSE.

**Skip this section if you did not select Option D.**

In [37]:
# Write your code here for Option D (if selected)
# Implement weighted sum prediction and evaluate



# --- Save your results ---
MAE_optionD, RMSE_optionD = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)

In [38]:
# Please don't change this cell

print("=" * 70)
print("Option D: Prediction Method Comparison")
print("=" * 70)
print(f"Weighted sum:           MAE = {MAE_optionD},  RMSE = {RMSE_optionD}")
print(f"Baseline (KNN-regr):    MAE = {MAE_task1},  RMSE = {RMSE_task1}")

Option D: Prediction Method Comparison
Weighted sum:           MAE = 3.5359,  RMSE = 3.719341339538494
Baseline (KNN-regr):    MAE = 3.5359,  RMSE = 3.719341339538494


---
# Summary of All Results

In [39]:
# Please don't change this cell

print("=" * 70)
print("SUMMARY")
print("=" * 70)
print()
print(f"Task 1 Baseline:  MAE = {MAE_task1},  RMSE = {RMSE_task1}")
print()
print(f"Selected options: {OPTION_1}, {OPTION_2}")
print()

if OPTION_1 == 'A' or OPTION_2 == 'A':
    print("Option A — Similarity Measure Comparison:")
    print(f"  Cosine:           MAE = {MAE_optionA_cosine},  RMSE = {RMSE_optionA_cosine}")
    print(f"  Pearson:          MAE = {MAE_optionA_pearson},  RMSE = {RMSE_optionA_pearson}")
    print()

if OPTION_1 == 'B' or OPTION_2 == 'B':
    print("Option B — Neighbourhood Size Tuning:")
    print(f"  k =  5:  MAE = {MAE_optionB_k5},  RMSE = {RMSE_optionB_k5}")
    print(f"  k = 10:  MAE = {MAE_optionB_k10},  RMSE = {RMSE_optionB_k10}")
    print(f"  k = 20:  MAE = {MAE_optionB_k20},  RMSE = {RMSE_optionB_k20}")
    print(f"  k = 30:  MAE = {MAE_optionB_k30},  RMSE = {RMSE_optionB_k30}")
    print(f"  k = 50:  MAE = {MAE_optionB_k50},  RMSE = {RMSE_optionB_k50}")
    print()

if OPTION_1 == 'C' or OPTION_2 == 'C':
    print("Option C — Significance Weighting:")
    print(f"  T = 10:  MAE = {MAE_optionC_T10},  RMSE = {RMSE_optionC_T10}")
    print(f"  T = 20:  MAE = {MAE_optionC_T20},  RMSE = {RMSE_optionC_T20}")
    print(f"  T = 30:  MAE = {MAE_optionC_T30},  RMSE = {RMSE_optionC_T30}")
    print(f"  T = 50:  MAE = {MAE_optionC_T50},  RMSE = {RMSE_optionC_T50}")
    print()

if OPTION_1 == 'D' or OPTION_2 == 'D':
    print("Option D — Prediction Method Comparison:")
    print(f"  Weighted sum:     MAE = {MAE_optionD},  RMSE = {RMSE_optionD}")
    print()

SUMMARY

Task 1 Baseline:  MAE = 3.5359,  RMSE = 3.719341339538494

Selected options: A, B

Option A — Similarity Measure Comparison:
  Cosine:           MAE = 3.5359,  RMSE = 3.719341339538494
  Pearson:          MAE = 3.5359,  RMSE = 3.719341339538494

Option B — Neighbourhood Size Tuning:
  k =  5:  MAE = 3.5359,  RMSE = 3.719341339538494
  k = 10:  MAE = 3.5359,  RMSE = 3.719341339538494
  k = 20:  MAE = 3.5359,  RMSE = 3.719341339538494
  k = 30:  MAE = 3.5359,  RMSE = 3.719341339538494
  k = 50:  MAE = 3.5359,  RMSE = 3.719341339538494

